In [ ]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
import requests
import time

In [ ]:
import sys
from pathlib import Path

for candidato in (Path.cwd(), *Path.cwd().resolve().parents):
    if (candidato / "src").is_dir():
        if str(candidato) not in sys.path:
            sys.path.insert(0, str(candidato))
        break

from src.paths import RAIZ, DIR_RAW, DIR_PROCESSED, DIR_REPORTS

PATH_PROCESSED = DIR_PROCESSED / "df_final_zonas_temperaturas_2023_2026.parquet"

In [ ]:
# 1. Releer el archivo Parquet recién creado
df_t = pd.read_parquet(PATH_PROCESSED, engine="pyarrow")
df_t.columns

In [ ]:
fecha_min = df_t['fecha'].min()
fecha_max = df_t["fecha"]. max()
len(pd.date_range(fecha_min, fecha_max))

In [ ]:
print(fecha_min, fecha_max)

In [ ]:
PATH_demanda = DIR_PROCESSED / "demanda_horaria.parquet"
df_d = pd.read_parquet(PATH_demanda, engine="pyarrow")
df_d.columns

In [ ]:
df_d

In [ ]:
df_t

In [ ]:
# Construcción de la clave con casteo explícito a datetime64[ns]
df_d['fecha'] = (
    df_d['datetime_utc']
    .dt.tz_convert("Europe/Madrid")
    .dt.normalize()
    .dt.tz_localize(None)
    .astype("datetime64[ns]")
)

# 1. Comprobar dtype exacto
print("Dtype:", df_d['fecha'].dtype)
assert df_d['fecha'].dtype == 'datetime64[ns]', "El tipo de dato debe ser datetime64[ns]"

# 2. Número de fechas únicas
print("Fechas únicas:", df_d['fecha'].nunique())

# 3. Distribución de registros por día
print("\nFrecuencia de horas por día:")
print(df_d['fecha'].value_counts().value_counts())

# 4. Rango de fechas
print("\nFecha mínima:", df_d['fecha'].min())
print("Fecha máxima:", df_d['fecha'].max())

In [ ]:
df = df_d.merge(df_t, on="fecha", how="left", validate="m:1")

In [ ]:
assert df.shape == (30648, 13), df.shape

# 48 NaN, todos en las dos columnas de guadalquivir
print(df.isna().sum())

# y todos del mismo día
fechas_nan = df.loc[df['tmin_guadalquivir'].isna(), 'fecha'].unique()
print(fechas_nan)

# difusión: un día cualquiera
dia = df[df['fecha'] == '2024-05-14']
print(len(dia), dia['tmin_continental'].nunique())

In [ ]:
df = df[df['fecha'] != '2023-12-15'].reset_index(drop=True)
assert df.shape == (30624, 13), df.shape
assert df.isna().sum().sum() == 0, "Quedan NaN"